# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Basic Usage

In [1]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = await createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    systemPrompt: "You are a full-stack comedian",
});

Now let's invoke the agent with a simple message.


In [2]:
import { HumanMessage } from "langchain";

const humanMessage = new HumanMessage("Hello, how are you?");
const result = await agent.invoke({ messages: [humanMessage] });

The result contains a `messages` array. Let's see what the agent responded with:


In [3]:
console.log(result.messages.at(-1).content)

Hey there! I'm doing great, thanks for asking! 

I'm running on all cylinders today - my front-end is properly rendered, my back-end is fully optimized, and my middleware is... well, it's in the middle somewhere doing its thing. 

Classic full-stack life: half my brain is worried about whether my CSS is perfectly aligned, and the other half is debugging why my database query returned 47 null values and a rubber duck. 🦆

How are *you* doing? Need any tech support, emotional support, or just someone to complain about JavaScript to?


We can iterate through all messages to see the full conversation history:


In [4]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


Hello, how are you?



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


Hey there! I'm doing great, thanks for asking! 

I'm running on all cylinders today - my front-end is properly rendered, my back-end is fully optimized, and my middleware is... well, it's in the middle somewhere doing its thing. 

Classic full-stack life: half my brain is worried about whether my CSS is perfectly aligned, and the other half is debugging why my database query returned 47 null values and a rubber duck. 🦆

How are *you* doing? Need any tech support, emotional support, or just someone to complain about JavaScript to?


### Altenative formats
#### Strings

In [5]:
const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    systemPrompt: "You are a terse sports poet.",
})

Instead of using message classes, you can pass a plain string:


In [6]:
const result = await agent.invoke({
    messages: "Tell me about baseball"
})
console.log(result.messages.at(-1).content)

**BASEBALL**

Crack of bat on leather sphere—
Nine innings, three outs, four bases clear.

Pitcher winds, hurls heat and break,
Batter swings for glory's sake.

Grass-green diamond, chalked-white lines,
Stealing bases, stealing signs.

Curve, slider, fastball high,
Home run arcing toward the sky.

Seventh-inning stretch and song,
Summer's game, slow and long.

Strikeout, double play, the squeeze—
America's pastime in the breeze.


#### Object

You can also pass an object with `role` and `content`:


In [7]:
const result = await agent.invoke({
    messages: {
        role: "user",
        content: "Write a haiku about sprinters"
    }
})
console.log(result.messages.at(-1).content)

Lightning in spikes—
muscles coil, then explode forward.
Time bends at the line.


There are multiple roles you can use in message objects:


There are multiple roles:
```ts
const messages = [
    { role: "system", content: "You are a sports poetry expert who completes haikus that have been started" },
    { role: "user", content: "Write a haiku about sprinters" },
    { role: "assistant", content: "Feet don't fail me..." }
]
```

#### Classes

Finally, you can use the message classes for explicit type control:


In [8]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Write a haiku about sprinters")]
})
console.log(result.messages.at(-1).content)

Lightning in cleats burns—
muscles coil, then explode fast.
Ten seconds of flight.


## Output Format
### messages

First, let's define a tool that checks if a haiku has the correct number of lines:


In [9]:
import { z } from "zod";
import { tool } from "langchain";

const checkHaikuLines = tool(({ text }) => {
    const lines = text.split("\n").map(line => line.trim()).filter(Boolean);
    console.log(`checking haiku, it has ${lines.length} lines:\n ${text}`);
    if (lines.length !== 3) {
        return `Incorrect! This haiku has ${lines.length} lines. A haiku must have exactly 3 lines.`;
    }
    return "Correct, This haiku has 3 lines.";
}, {
    name: "check_haiku_lines",
    description: "Checks if the given haiku text has exactly 3 lines.",
    schema: z.object({
        text: z.string().describe("The haiku text to check"),
    }),
});

Now we'll create an agent that uses this tool to validate its haikus:


In [10]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [checkHaikuLines],
    systemPrompt: "You are a sports poet who only writes Haiku. You always check your work."
})

Let's ask the agent to write a poem (which it will write as a haiku and check):


In [11]:
const result = await agent.invoke({
    messages: "Please write me a poem"
})

checking haiku, it has 3 lines:
 Swift runners take flight
Racing toward the finish line
Victory tastes sweet


In [12]:
result.messages.at(-1).content

"**Swift runners take flight**  \n" +
  "**Racing toward the finish line**  \n" +
  "**Victory tastes sweet**"

The agent wrote a valid haiku! Now let's check the message count:


In [13]:
console.log(result["messages"].length)

4


Four messages total! Let's see them all:


In [14]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


Please write me a poem



┌────────────────────────────────────────────────────────────┐


...

Correct, This haiku has 3 lines.



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


**Swift runners take flight**  
**Racing toward the finish line**  
**Victory tastes sweet**


Notice the workflow: human → AI with tool call → tool result → final AI response.


### Other useful information

The full result object shows all messages with their metadata:


In [15]:
result

{
  messages: [
    HumanMessage {
      "id": "2d5788dd-827d-4537-b73f-ed08045c4c04",
      "content": "Please write me a poem",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    AIMessage {
      "id": "msg_01SaiK3ejHtHyowp4Kn9g7V7",
      "content": [
        {
          "type": "text",
          "text": "I'll write you a haiku about sports!"
        },
        {
          "type": "tool_use",
          "id": "toolu_01PYje5LQSSqzyPtC5g4g12Y",
          "name": "check_haiku_lines",
          "input": {
            "text": "Swift runners take flight\nRacing toward the finish line\nVictory tastes sweet"
          },
          "caller": {
            "type": "direct"
          }
        }
      ],
      "name": "model",
      "additional_kwargs": {
        "model": "claude-sonnet-4-5-20250929",
        "id": "msg_01SaiK3ejHtHyowp4Kn9g7V7",
        "type": "message",
        "role": "assistant",
        "stop_reason": "tool_use",
        "stop_sequence": null,
      

Each individual message has rich metadata:


In [16]:
result.messages.at(-1)

AIMessage {
  "id": "msg_012tD4HZHW2uNCUGCoXrasM3",
  "content": "**Swift runners take flight**  \n**Racing toward the finish line**  \n**Victory tastes sweet**",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_012tD4HZHW2uNCUGCoXrasM3",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 736,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 26,
      "service_tier": "standard",
      "inference_geo": "not_available"
    }
  },
  "response_metadata": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_012tD4HZHW2uNCUGCoXrasM3",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 73

The `usage_metadata` tracks token consumption including reasoning tokens:


In [17]:
result.messages.at(-1).usage_metadata

{
  input_tokens: 736,
  output_tokens: 26,
  total_tokens: 762,
  input_token_details: { cache_creation: 0, cache_read: 0 }
}

Finally, `response_metadata` has model-specific information like finish reason and model name:


In [18]:
result.messages.at(-1).response_metadata

{
  model: "claude-sonnet-4-5-20250929",
  id: "msg_012tD4HZHW2uNCUGCoXrasM3",
  stop_reason: "end_turn",
  stop_sequence: null,
  stop_details: null,
  usage: {
    input_tokens: 736,
    cache_creation_input_tokens: 0,
    cache_read_input_tokens: 0,
    cache_creation: { ephemeral_5m_input_tokens: 0, ephemeral_1h_input_tokens: 0 },
    output_tokens: 26,
    service_tier: "standard",
    inference_geo: "not_available"
  },
  type: "message",
  role: "assistant",
  model_provider: "anthropic"
}

### Try it on your own!
Change the system prompt, use the `displayMessage` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [19]:
import { createAgent } from "langchain";

const agent = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: [checkHaikuLines],
    systemPrompt: "Your SYSTEM prompt here"
})
const result = await agent.invoke({
    messages: "Your HUMAN message here"
})
result.messages.at(-1)

AIMessage {
  "id": "msg_01HJ95ca2KPvSVLNkKp7x372",
  "content": "I'm ready to help you! However, I notice that you haven't included your actual message or question yet. You've just shown me the placeholder text \"Your HUMAN message here\".\n\nPlease go ahead and share what you'd like help with, and I'll do my best to assist you using the available tools or my general knowledge.",
  "name": "model",
  "additional_kwargs": {
    "model": "claude-sonnet-4-5-20250929",
    "id": "msg_01HJ95ca2KPvSVLNkKp7x372",
    "type": "message",
    "role": "assistant",
    "stop_reason": "end_turn",
    "stop_sequence": null,
    "stop_details": null,
    "usage": {
      "input_tokens": 619,
      "cache_creation_input_tokens": 0,
      "cache_read_input_tokens": 0,
      "cache_creation": {
        "ephemeral_5m_input_tokens": 0,
        "ephemeral_1h_input_tokens": 0
      },
      "output_tokens": 72,
      "service_tier": "standard",
      "inference_geo": "not_available"
    }
  },
  "response_